In [83]:

! pip install -q youtube-transcript-api langchain-community faiss-cpu langchain-google-genai langchain-groq python-dotenv

In [84]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_groq import ChatGroq

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

In [85]:
def get_transcript(video_id):
  transcript_obj=YouTubeTranscriptApi().fetch(video_id)
  return " ".join(snippet.text for snippet in transcript_obj)

In [86]:
transcript = get_transcript( "fFMcolWgD3w" )
print(transcript)

August 1991, Barrow Neurological
Institute in Phoenix, USA. In the operating room of this hospital, a 35-year-old woman, Pam
Reynolds, was lying down. Pam's brain had a tumor
that could explode at any time. The doctors told Pam that
she had very little time left. There was only one way to save her. An extremely risky operation. An operation in which the her heartbeats had to be completely stopped by machines. And then her skull would be cut
open to extract all the blood and fluids. It sounds scary, but the doctors had no other option. They began the operation immediately. Pam's eyes were taped, and small speakers were
placed on both her ears, they continuously played clicking
sounds to check the brain's response. With one click, the
machine was switched on, and Pam's heartbeat stopped. There was no response from her brain. No activity, no brainwaves. Nothing that could be detected
and to say that Pam was still alive. Because according to every
medical science definition, Pam had died b

In [87]:
splitter = RecursiveCharacterTextSplitter( chunk_size = 1000 , chunk_overlap = 200)

In [88]:
chunks = splitter .create_documents([transcript])

In [89]:
len(chunks)

29

In [90]:
from dotenv import load_dotenv

load_dotenv()

True

In [91]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.index_to_docstore_id

{0: 'a42bf7bd-23b2-496f-9d3f-e7981f7dc94c',
 1: 'afcd6860-47f9-4e5c-a8ac-dc8724ef1522',
 2: 'a6142786-3b26-4ed3-8295-ed7692b80e16',
 3: '1eec14a2-574b-45f2-bb86-76782b6fd62f',
 4: 'e8d8a3a7-265e-4af3-a383-142ff553a016',
 5: '164a5cab-6e46-41f5-a085-23f55fd41520',
 6: '9efad25a-be2d-42f5-9d3d-6062d5ba5d49',
 7: 'af925a95-b96a-4fc4-999c-4f0e008c2d8a',
 8: 'a7141504-6a7f-4d9d-b9a3-4f717e179d1d',
 9: 'a3759d0c-e34e-41bb-b531-69edd989ccf4',
 10: '3c984738-1d62-4074-ad98-6830de9cfdd2',
 11: '260f30f5-5252-46ee-a3a7-d05215d56b48',
 12: '8234fa41-c84a-4711-942f-43fd8e863f16',
 13: 'cca6358b-7ba0-47e6-8219-7a8945faaa4a',
 14: '6ed39a3d-bdca-44b9-ab5d-b39058710ea0',
 15: '1ffcf48d-3a77-4811-bba6-e57598d92c0e',
 16: '3526f046-016f-4c36-b47f-42f737389be7',
 17: '844b0181-213e-44d9-b066-60288cc6e407',
 18: 'fdf47269-cd0e-4e03-b303-16358a5dc35a',
 19: '206510e8-be1f-41f4-8f5b-9db3cd280a8c',
 20: 'ce7161d5-cfe1-426c-8326-76640a98db31',
 21: '050e5951-a1a6-4bb4-b678-addeacd536ca',
 22: 'dca1e3d6-d7fc-

In [92]:
vector_store.get_by_ids(['13a453f2-f90a-45ce-9368-4791289fd26a'])

[]

In [93]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2) 
prompt = PromptTemplate(    
    template="""      
      You are a helpful assistant.      
      Answer ONLY from the provided transcript context.      
      If the context is insufficient, just say you don't know.      
      
      {context}      
      Question: {question}    
    """,    
    input_variables = ['context', 'question'] 
)

In [94]:
retriever = vector_store.as_retriever(search_type="similarity", 
search_kwargs={"k": 4})

In [96]:
question = "is the topic discussed in this video? if yes then what was discussed" 
retrieved_docs = retriever.invoke(question)

In [ ]:
retriever.invoke('what happen after death')

[Document(id='a2338c6b-8f1c-43a2-b946-32cdc620c52c', metadata={}, page_content="them, his legs had been cut. There were no such marks before. Hahaha! In India, there are many\nsuch stories of afterlife. In them, the Yamdut takes a person, and then presents him to a man with a book, who keeps a record of every person. Then it turns out that\nsomeone made a mistake, and the wrong man was kidnapped. So he is pushed back. Now, this sounds like a story\ntaken from a religious book. In Hinduism, the journey after death\nis mentioned in the Garud Purana. According to this, when\nthe Yamalok reaches, a secret that has\nbeen written all his life. Yama Raj decides what will happen to\nsomeone else after reading this letter. The question is, how much truth\nis there in these people's stories? How much can we believe in their stories? Science doesn't believe in religious books. Nor does it believe in\nthese anecdotal cases. For science, these stories are just\na collection that cannot be verified.

In [100]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)


In [101]:
context_text

'as well Financial freedom your choice of work and\nthe influence of reaching out to do what they\nlike Influence So to teach you this I have created\na 7 hour course on it The YouTube Blueprint. It\'s not a generic course. I teach you step by step, how to make a successful\nYouTube channel in the long term. Where to get video ideas from, how to change your story in the script, how to confidently speak\nin front of the camera, and how to crack the YouTube algorithm. I have taught everything in\nmy 12 years of experience. And the people who have taken this course, their average rating is 4.9 out of 5. You can see some of their reviews. I finished my course today. I liked it a lot. I learned a lot of things\nlike ideas for niche topics. All the aspects which can\nbe there for a YouTuber are there in terms of how to grow, how to earn, what are the stresses, what are the challenges. And the best part was\nthat you have completely connected your life\n\nmaking our own Earth heaven. Because 

In [103]:
{"type":"string"}

{'type': 'string'}

In [104]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [105]:
final_prompt

StringPromptValue(text='      \n      You are a helpful assistant.      \n      Answer ONLY from the provided transcript context.      \n      If the context is insufficient, just say you don\'t know.      \n\n      as well Financial freedom your choice of work and\nthe influence of reaching out to do what they\nlike Influence So to teach you this I have created\na 7 hour course on it The YouTube Blueprint. It\'s not a generic course. I teach you step by step, how to make a successful\nYouTube channel in the long term. Where to get video ideas from, how to change your story in the script, how to confidently speak\nin front of the camera, and how to crack the YouTube algorithm. I have taught everything in\nmy 12 years of experience. And the people who have taken this course, their average rating is 4.9 out of 5. You can see some of their reviews. I finished my course today. I liked it a lot. I learned a lot of things\nlike ideas for niche topics. All the aspects which can\nbe there for 

In [106]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes. The video discusses several related topics:

- **The after‑life and the “21‑gram” experiment** – It recounts Dr. Duncan MacDougall’s 1901 attempt to weigh a human at the moment of death, claiming the body lost about 21 grams when the soul supposedly left the body.  
- **Regrets of people who are dying** – It shares the findings of Bronnie Ware, a palliative‑care nurse in Australia, who asked dying patients about their biggest regrets. The most common answer was: “I wish I had lived the life I wanted to live, not the life others expected of me,” and “I wish I had allowed myself to be happier.”  
- **Advice for living a fulfilling life** – The speaker urges viewers to try new things, learn new skills, and pursue their dreams, emphasizing that living true to oneself leads to fewer regrets.  
- **YouTube as a path to financial freedom** – The narrator ties the discussion to his own experience, promoting his 7‑hour “YouTube Blueprint” course (coupon code LIFE42) as a way to achieve fin